# Kittiwake population dynamics and quasi-extinction risk

This executed notebook presents the validated data, fitted-model evidence, chronological holdout and corrected risk scenario. Reusable logic is in `population_risk/`; the notebook reads the exact artifacts used by the dashboard.

**Boundary:** 36 historical observations, 1949–1984. Results describe a conditional local-population scenario, not current global species risk.


In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from population_risk.data import load_population

data = load_population(ROOT / 'data/kittiwake_gpdd_6633.xlsx')
metadata = json.loads((ROOT / 'artifacts/metadata.json').read_text())
print(data.to_string(index=False))
print(f"Observations: {len(data)}; window: {int(data.SampleYear.min())}–{int(data.SampleYear.max())}")


 SampleYear  Population
       1949           8
       1950          22
       1951          24
       1952          27
       1953          32
       1954          41
       1955          68
       1956          70
       1957          70
       1958          77
       1959          89
       1960         104
       1961         123
       1962         145
       1963         166
       1964         176
       1965         208
       1966         194
       1967         206
       1968         193
       1969         181
       1970         183
       1971         169
       1972         169
       1973         159
       1974         145
       1975         144
       1976         137
       1977         135
       1978         149
       1979         143
       1980         173
       1981         179
       1982         141
       1983         175
       1984         138
Observations: 36; window: 1949–1984


## Deterministic evidence

Exponential, logistic and Gompertz curves are fit under a common Gaussian residual likelihood. AICc is used because the sample is small. Good fit does not identify an ecological mechanism.

![Deterministic fits](../artifacts/figures/deterministic_fits.png)


In [2]:
deterministic = pd.read_csv(ROOT / 'artifacts/deterministic_models.csv')
print(deterministic[['model', 'aicc', 'bic', 'rmse', 'akaike_weight']].round(4).to_string(index=False))


      model     aicc      bic    rmse  akaike_weight
   Logistic 328.9429 333.9866 20.5052         0.9437
   Gompertz 334.5814 339.6252 22.1756         0.0563
Exponential 381.0131 385.0136 43.7793         0.0000


## Stochastic transition evidence

All stochastic candidates use the same Euler-logistic conditional mean. Their variance scales as population, squared population, or both. These labels describe equations rather than proven causes.


In [3]:
stochastic = pd.read_csv(ROOT / 'artifacts/stochastic_models.csv')
columns = ['model', 'growth_rate', 'carrying_capacity', 'sigma_demographic', 'sigma_environmental', 'aicc', 'akaike_weight']
print(stochastic[columns].round(4).to_string(index=False))


        model  growth_rate  carrying_capacity  sigma_demographic  sigma_environmental     aicc  akaike_weight
  Demographic       0.3397           168.5814             1.4325                  NaN 293.4725         0.7824
Environmental       0.5044           163.5253                NaN               0.2642 337.3463         0.0000
     Combined       0.3397           168.5816             1.4325               0.0000 296.0316         0.2176


## Chronological validation

The first 27 observations are fitted; the final nine are tested. Persistence repeats the previous observation. This asks a different question from full-data AICc.


In [4]:
validation = pd.read_csv(ROOT / 'artifacts/holdout_validation.csv')
print(validation.round(4).to_string(index=False))


        model      mae     rmse  mean_log_score
Environmental  18.2015  20.7946          4.8254
     Combined  19.0256  22.5084          4.6029
  Demographic  19.0256  22.5084          4.6029
  Persistence  19.3333  24.0139             NaN
     Logistic  27.4476  32.2953             NaN
     Gompertz  29.9186  34.3405             NaN
  Exponential 118.7567 123.1098             NaN


## Corrected quasi-extinction scenario

The original notebook starts at the first value (8) while setting quasi-extinction to ≤10, but checks only after taking a step. The corrected primary scenario starts from the final observed population (138 in 1984), uses 20,000 paths and treats threshold crossing as absorbing.


In [5]:
risk = pd.read_csv(ROOT / 'artifacts/extinction_risk.csv')
primary = risk[(risk.horizon == 30) & (risk.threshold == 10)]
print(primary[['model', 'events', 'simulations', 'probability', 'mc_ci_low', 'mc_ci_high']].round(6).to_string(index=False))


        model  events  simulations  probability  mc_ci_low  mc_ci_high
  Demographic     0.0        20000      0.00000   0.000000    0.000192
Environmental   391.0        20000      0.01955   0.017721    0.021563
     Combined     0.0        20000      0.00000   0.000000    0.000192
AICc-weighted     NaN        20000      0.00000        NaN         NaN


![Risk sensitivity](../artifacts/figures/risk_sensitivity.png)

Zero simulated events do not prove zero risk; the finite-run upper interval remains positive. The AICc-weighted value averages only this candidate set and does not include parameter or broader structural uncertainty.

## Reproduce

```bash
python -m pip install -r requirements-dev.txt
python -m population_risk
python -m pytest -q
python -m streamlit run app.py
```

Read the [methodology](../docs/METHODOLOGY.md), [project audit](../docs/PROJECT_AUDIT.md), [data card](../docs/DATA_CARD.md), and [interview guide](../docs/INTERVIEW_GUIDE.md).
